<a href="https://www.kaggle.com/code/jatin2055/langsmith-rag-v2?scriptVersionId=265481689" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [187]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [188]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader  
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage

from langsmith import traceable

import os

In [189]:
api_key='your_api_key'

langsmith = 'your_api_key'

os.environ["LANGCHAIN_API_KEY"] = langsmith
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="LangSmith - RAG v1"

os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'


In [190]:
model = ChatOpenAI(api_key=api_key)

# LOADER

In [191]:
@traceable(name="load_pdf_documents", tags=['pypdf', 'pdf_loader'], metadata={'loader': 'pypdfloader', 'task':'load_pdf'})
def load_pdf_documents(path):
    all_documents = []
    
    for path in PDF_PATH:
        print(path)
        loader = PyPDFLoader(path)
        all_documents.extend(loader.lazy_load())

    return all_documents

# Creating Chunks

In [192]:
@traceable(name="recursive_split_text", tags=['text_split', 'RecursiveCharacterTextSplitter'], metadata={'splitter': 'RecursiveCharacterTextSplitter', 'task':'split_text'})
def recursive_split_text(chunk_size, chunk_overlap, all_documents):
    splitter = RecursiveCharacterTextSplitter(chunk_size =chunk_size, chunk_overlap=chunk_overlap)
    pdf_split = splitter.split_documents(all_documents)
    return pdf_split

# Embedding

In [193]:

@traceable(name="add_to_vector_store", tags=['embed', 'vector_store'], metadata={'embedding_model': 'OpenAIEmbeddings', 'task':'create_embedding', 'vector_store': 'FAISS'})
def add_to_vector_store(pdf_split):
    embed = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)
    vs = FAISS.from_documents(pdf_split, embed)
    return vs

@traceable(name="fetch_vectore_store")
def fetch_vectore_store(index_dir):
    embed = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)
    vs = FAISS.load_local(
        str(index_dir),
        embed,
        allow_dangerous_deserialization=True
    )
    return vs
    
@traceable(name="fetch_external_vector_store")
def fetch_external_vector_store(pdf_path, index_dir):
    print(index_dir)
    

    
    directory_path = index_dir
    
    if os.path.isdir(directory_path):
        print(f"The directory '{directory_path}' exists.")
        vs = fetch_vectore_store(index_dir)
    else:
        print(f"The directory '{directory_path}' does not exist.")
        all_documents = load_pdf_documents(pdf_path)
        split_text = recursive_split_text(chunk_size=800, chunk_overlap=100, all_documents=all_documents)
        vs = add_to_vector_store(split_text)
        vs.save_local(str(index_dir))


    #print(vs)
    # else:
    #     all_documents = load_pdf_documents(pdf_path)
    #     split_text = recursive_split_text(chunk_size=800, chunk_overlap=100, all_documents=all_documents)
    #     vs = add_to_vector_store(split_text)
    #     vs.save_local(str(index_dir))
    return vs
    



In [194]:
#result = retriever.invoke("confusion matrix")


# Prompt

In [195]:
# This works
message = [
    ("system","""Answer only from the provided context. If not found, reply back You dont know"""),
    ("human",""" question: {question}.\n\n context : {context}""")
]

# OR 
# This doesnot works - dont know why?
# message = [
#     SystemMessage(content="""Answer only from the provided context. If not found, reply back You dont know"""),
#     HumanMessage(content=""" question: {question}.\n\n context : {context}""")
# ]

prompt = ChatPromptTemplate.from_messages(message)

# Output Parser

In [196]:
str_parser = StrOutputParser()

# Chaining

In [197]:
    
def combine_retrieved_docs(embed_docs):
    return "\n".join(d.page_content for d in embed_docs)

@traceable(name="pdf_rag_full_run")
def setup_pipeline_and_query(pdf_path, question):
    index_dir = '/kaggle/working/vector_embedding'
    external_vector_store = fetch_external_vector_store(pdf_path, index_dir)
    
    retriever = external_vector_store.as_retriever(
        search_type="mmr", # Maximum Marginal Relevance 
        search_kwargs = {"k": 3, "lambda_mult": 0.5 } # lambda_mult [0-1] : 0 - most diverse, 1: most similar (same as similarity search) 
    )


    
    parallel = RunnableParallel({
        "context": retriever | RunnableLambda(combine_retrieved_docs),
        "question": RunnablePassthrough()
        
    })

    config = {
        'run_name': 'LangSmith - RAG V2',
        'tags':['ChatPromptTemplate', 'FAISS', 'retriever', "mmr", "RecursiveCharacterTextSplitter", "PyPDFLoader", "load", "lazy_load"],
        'metadata': {'model': 'openai', 'parser_used': 'str output'}
    }
    
    chain = parallel | prompt | model | str_parser

    return chain.invoke(q.strip(), config=config)

    

In [198]:
q = input("Ask you question: ")
print(q)
print(q.strip())

PDF_PATH = ["https://www.stat.berkeley.edu/~rabbee/s154/ISLR_First_Printing.pdf", "https://www.sas.upenn.edu/~fdiebold/NoHesitations/BookAdvanced.pdf"]

ans = setup_pipeline_and_query(PDF_PATH, q.strip())

Ask you question:  chapters of the books


chapters of the books
chapters of the books
/kaggle/working/vector_embedding
The directory '/kaggle/working/vector_embedding' exists.


In [199]:
print(ans)

I do not have information on the specific chapters of the book mentioned.
